# The `control` modifier

**Download this notebook - {nb-download}`control.ipynb`**

A `with control(ctrl):` block applies its quantum operations when the control
qubit is in state $|1\rangle$, and leaves the target unchanged when it is in
state $|0\rangle$. The control qubit can also be in a superposition.

Here we use a controlled `x` gate to prepare the Bell state
$(|00\rangle + |11\rangle)/\sqrt{2}$ and run it on the emulator.

In [1]:
import numpy as np

from guppylang import guppy
from guppylang.std.builtins import control, output
from guppylang.std.debug import state_output
from guppylang.std.quantum import h, measure, qubit, x

In [2]:
@guppy
def bell_pair() -> None:
    ctrl = qubit()
    target = qubit()
    h(ctrl)
    with control(ctrl):
        x(target)

    state_output("bell", ctrl, target)
    output("control", measure(ctrl).read())
    output("target", measure(target).read())

The Hadamard gate prepares the control in $(|0\rangle + |1\rangle)/\sqrt{2}$.
The controlled `x` flips the target only in the $|1\rangle$ branch, entangling
the two qubits. The two branches stay in superposition until measurement.

Use the statevector simulator so we can check the amplitudes before measurement.
We compare states up to a global phase. The measurements must always agree.

In [3]:
shots = (
    bell_pair.emulator(n_qubits=2)
    .statevector_sim()
    .with_seed(42)
    .with_shots(100)
    .run()
)

state = shots.partial_state_dicts()[0]["bell"].as_single_state()
expected = np.array([1, 0, 0, 1]) / np.sqrt(2)
assert np.isclose(abs(np.vdot(expected, state)), 1)
assert all(shot.as_dict()["control"] == shot.as_dict()["target"] for shot in shots)
print("Bell state verified; all 100 shots measured matching bits.")

Bell state verified; all 100 shots measured matching bits.
